# Mat & Comp – RN 5: Redes convolucionales
## Buscar trazos en cada pedacito de la imagen

**Curso:** Redes neuronales (Sección 4) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** RN 4

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Calcular a mano una **convolución**: deslizar un **filtro** de 3 × 3 sobre una imagen y obtener un **mapa de características**.
2. Explicar qué detectan algunos filtros sencillos (bordes verticales, horizontales, desenfoque).
3. Aplicar **max pooling** y calcular el tamaño de la salida de cada capa.
4. Construir con PyTorch una **red convolucional** con `nn.Conv2d` y `nn.MaxPool2d`, y entrenarla con MNIST.
5. Comparar la red convolucional con la red de RN 4 en **parámetros**, **exactitud** y **robustez** cuando el dígito se corre.

## 🔍 ¿Cuántos pesos para buscar una raya?

En RN 4 la primera capa de la red tenía $784 \times 128 = 100\,352$ pesos: cada neurona tenía **un peso distinto para cada píxel**. Si una neurona aprendía a reconocer «una raya vertical en el centro», esa neurona no servía para una raya vertical **un poco más a la derecha**. Por eso la red se confundía al correr el dígito.

Imagina, en cambio, una **lupa** de 3 × 3 píxeles que sabe reconocer una raya vertical. La pasamos por **todas** las posiciones de la imagen y anotamos dónde la encuentra.

✏️ *¿Cuántos pesos tiene esa lupa? ¿Cuántas posiciones distintas puede ocupar una lupa de 3 × 3 dentro de una imagen de 28 × 28 (sin salirse)? ¿Necesita la lupa pesos distintos para cada posición?*

**Mi respuesta:**

Esa lupa se llama **filtro**, y pasarla por toda la imagen se llama **convolución**. Hoy construiremos redes que **aprenden** sus propios filtros.

## 1. La convolución a mano

Empecemos con una imagen diminuta de 6 × 6 que tiene una raya vertical, y un filtro de 3 × 3 que busca **bordes verticales**: le gustan los píxeles claros a la izquierda y oscuros a la derecha… o al revés, según el signo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

imagen = np.array([
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 0],
], dtype=float)

filtro_vertical = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1],
], dtype=float)

**La regla:** ponemos el filtro sobre un pedazo de 3 × 3 de la imagen, multiplicamos casilla por casilla y **sumamos** los 9 productos. Ese número va a la salida. Luego corremos el filtro una casilla y repetimos.

Para el pedazo de arriba a la izquierda (filas 0–2, columnas 0–2):

$$\begin{bmatrix}0&0&1\\0&0&1\\0&0&1\end{bmatrix} \cdot \begin{bmatrix}1&0&-1\\1&0&-1\\1&0&-1\end{bmatrix} \;\to\; 0+0-1+0+0-1+0+0-1 = -3$$

✏️ *Antes de ejecutar: calcula a mano el número para el pedazo de las columnas 1–3 y para el de las columnas 2–4.*

In [ ]:
def convolucion(img, filtro):
    k = filtro.shape[0]
    alto = img.shape[0] - k + 1
    ancho = img.shape[1] - k + 1
    salida = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            pedazo = img[i:i+k, j:j+k]
            salida[i, j] = (pedazo * filtro).sum()
    return salida

mapa = convolucion(imagen, filtro_vertical)
print(mapa)

La salida tiene 4 × 4 casillas: el filtro de 3 × 3 cabe en 4 posiciones por fila. En general, una imagen de $n \times n$ con un filtro de $k \times k$ da

$$\text{salida} = n - k + 1.$$

El resultado se llama **mapa de características**: dice **dónde** encontró el filtro lo que busca. Aquí vale $-3$ en el borde izquierdo de la raya (oscuro → claro), $+3$ en el borde derecho (claro → oscuro) y $0$ donde todo es igual.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(10, 3.2))
ejes[0].imshow(imagen, cmap="gray_r"); ejes[0].set_title("imagen 6 × 6")
ejes[1].imshow(filtro_vertical, cmap="RdBu", vmin=-1, vmax=1); ejes[1].set_title("filtro 3 × 3")
ejes[2].imshow(mapa, cmap="RdBu", vmin=-3, vmax=3); ejes[2].set_title("mapa de características 4 × 4")
for eje in ejes:
    eje.set_xticks([]); eje.set_yticks([])
plt.tight_layout()
plt.show()

✏️ *Ahora gira la raya: haz una imagen con una raya **horizontal** y aplícale el mismo `filtro_vertical`. ¿Qué mapa esperas? Compruébalo.*

In [ ]:
imagen_h = imagen.T          # la transpuesta: la raya queda horizontal
print(convolucion(imagen_h, filtro_vertical))

¡Puros ceros! El filtro vertical **no ve** las rayas horizontales. Cada filtro se especializa en un tipo de trazo.

## 2. Filtros sobre un dígito de verdad

Carguemos MNIST, como en RN 4, y apliquemos cuatro filtros a un dígito.

In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

entreno = datasets.MNIST("datos", train=True, download=True, transform=transforms.ToTensor())
prueba = datasets.MNIST("datos", train=False, download=True, transform=transforms.ToTensor())

digito = entreno[0][0][0].numpy()     # un 5, como arreglo de 28 × 28
print("Forma:", digito.shape, "  etiqueta:", entreno[0][1])

In [ ]:
filtros = {
    "bordes verticales": np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]]),
    "bordes horizontales": np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]]),
    "diagonal": np.array([[0, 1, 1], [-1, 0, 1], [-1, -1, 0]]),
    "desenfoque": np.ones((3, 3)) / 9,
}

fig, ejes = plt.subplots(2, 5, figsize=(14, 6))
ejes[0, 0].imshow(digito, cmap="gray_r"); ejes[0, 0].set_title("dígito 28 × 28")
ejes[1, 0].axis("off")
for col, (nombre, f) in enumerate(filtros.items(), start=1):
    m = convolucion(digito, f)
    ejes[0, col].imshow(f, cmap="RdBu", vmin=-1, vmax=1)
    ejes[0, col].set_title(nombre)
    lim = np.abs(m).max()
    ejes[1, col].imshow(m, cmap="RdBu", vmin=-lim, vmax=lim)
    ejes[1, col].set_title(f"mapa {m.shape[0]} × {m.shape[1]}")
for eje in ejes.flat:
    eje.set_xticks([]); eje.set_yticks([])
plt.tight_layout()
plt.show()

Cada mapa resalta una cosa distinta: el primero los lados izquierdo y derecho de los trazos, el segundo los bordes de arriba y de abajo, el tercero los trazos inclinados, y el último una versión borrosa del dígito.

✏️ *¿Por qué los mapas miden 26 × 26 y no 28 × 28? Usa la fórmula $n - k + 1$.*

### Relleno (padding)

Si queremos que la salida mida lo mismo que la entrada, podemos poner un **marco de ceros** alrededor de la imagen. Con un marco de ancho $p$:

$$\text{salida} = n + 2p - k + 1.$$

Con $n = 28$, $k = 3$ y $p = 1$, la salida mide $28 + 2 - 3 + 1 = 28$.

## 3. Max pooling: resumir y tolerar pequeños movimientos

Después de la convolución se suele **resumir** el mapa: lo partimos en cuadritos de 2 × 2 y de cada uno nos quedamos con el **máximo**. Esto se llama **max pooling**. El mapa queda con la mitad de alto y la mitad de ancho.

In [ ]:
def max_pooling(mapa, t=2):
    alto, ancho = mapa.shape[0] // t, mapa.shape[1] // t
    salida = np.zeros((alto, ancho))
    for i in range(alto):
        for j in range(ancho):
            salida[i, j] = mapa[i*t:(i+1)*t, j*t:(j+1)*t].max()
    return salida

ejemplo = np.array([
    [1, 3, 0, 2],
    [4, 2, 1, 0],
    [0, 0, 5, 1],
    [1, 2, 3, 2],
], dtype=float)
print(max_pooling(ejemplo))

✏️ *Comprueba a mano los cuatro números: el máximo de cada cuadrito de 2 × 2.*

¿Para qué sirve? El pooling dice «**en esta zona** apareció el trazo», sin importar el píxel exacto. Veámoslo: corremos el dígito **1 píxel** a la derecha y comparamos los mapas de bordes verticales antes y después del pooling.

In [ ]:
movido = np.roll(digito, 1, axis=1)          # 1 píxel a la derecha
f = filtros["bordes verticales"]

m1, m2 = convolucion(digito, f), convolucion(movido, f)
p1, p2 = max_pooling(np.maximum(m1, 0)), max_pooling(np.maximum(m2, 0))   # ReLU y luego pooling
r1, r2 = np.maximum(m1, 0), np.maximum(m2, 0)

def parecido(a, b):
    """Qué tanto se parecen dos mapas: 1 = idénticos, 0 = nada en común."""
    return (a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum())

print(f"Parecido de los mapas sin pooling: {parecido(r1, r2):.2f}")
print(f"Parecido de los mapas con pooling: {parecido(p1, p2):.2f}")

Después del pooling, los mapas del dígito original y del dígito corrido se parecen **más**. Cada capa de pooling hace a la red un poco más tolerante a los movimientos pequeños.

## 4. La convolución en PyTorch: `nn.Conv2d`

PyTorch trae la convolución lista. `nn.Conv2d(canales_entrada, canales_salida, tamaño_filtro)` crea una capa con **varios filtros** a la vez: uno por cada canal de salida. Los filtros empiezan al azar y la red los **aprende** con el descenso de gradiente.

Primero comprobemos que hace lo mismo que nuestra función: le ponemos a mano nuestro filtro de bordes verticales.

In [ ]:
capa = nn.Conv2d(1, 1, kernel_size=3, bias=False)
with torch.no_grad():
    capa.weight[0, 0] = torch.tensor(filtros["bordes verticales"], dtype=torch.float32)

x = torch.tensor(digito).reshape(1, 1, 28, 28)    # (lote, canales, alto, ancho)
salida_torch = capa(x)[0, 0].detach().numpy()
salida_nuestra = convolucion(digito, filtros["bordes verticales"])

print("Forma de la salida:", tuple(capa(x).shape))
print("¿Iguales?", np.allclose(salida_torch, salida_nuestra, atol=1e-5))

¡Idénticas! Fíjate en la forma `(1, 1, 26, 26)`: PyTorch siempre trabaja con **cuatro** dimensiones: (imágenes del lote, canales, alto, ancho).

### ¿Cuántos parámetros tiene una capa convolucional?

Cada filtro tiene $k \times k$ pesos **por cada canal de entrada**, más un sesgo:

$$\text{parámetros} = \text{filtros} \times (\text{canales de entrada} \times k \times k + 1).$$

Con 16 filtros de 3 × 3 sobre una imagen gris (1 canal): $16 \times (1 \times 9 + 1) = 160$. ¡No depende del tamaño de la imagen!

In [ ]:
capa16 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
print("Parámetros:", sum(p.numel() for p in capa16.parameters()))
print("Forma de los pesos:", tuple(capa16.weight.shape))
print("Salida para un lote de 64:", tuple(capa16(torch.zeros(64, 1, 28, 28)).shape))

La salida tiene **16 canales**: un mapa de características de 28 × 28 por cada filtro (con `padding=1` la imagen conserva su tamaño). La siguiente capa convolucional recibe esos 16 mapas y combina sus trazos para encontrar formas más complejas: esquinas, curvas, círculos…

## 5. Nuestra primera red convolucional

La red tendrá dos bloques «convolución → ReLU → pooling» y al final una capa `nn.Linear` que decide el dígito:

| Capa | Salida (por imagen) | Parámetros |
|---|---|---|
| entrada | 1 × 28 × 28 | — |
| `Conv2d(1, 16, 3, padding=1)` + ReLU | 16 × 28 × 28 | 16 · (1 · 9 + 1) = 160 |
| `MaxPool2d(2)` | 16 × 14 × 14 | 0 |
| `Conv2d(16, 32, 3, padding=1)` + ReLU | ✏️ | ✏️ |
| `MaxPool2d(2)` | ✏️ | 0 |
| `Flatten()` | ✏️ | 0 |
| `Linear(…, 10)` | 10 | ✏️ |

✏️ *Completa la tabla antes de ejecutar la celda de abajo. Ojo: la segunda convolución recibe 16 canales.*

In [ ]:
torch.manual_seed(0)
red_cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
)

# Seguimos una imagen capa por capa
x = torch.zeros(1, 1, 28, 28)
print(f"{'entrada':42} {tuple(x.shape)}")
for capa in red_cnn:
    x = capa(x)
    n = sum(p.numel() for p in capa.parameters())
    print(f"{str(capa):42} {str(tuple(x.shape)):18} {n:6} parámetros")

total_cnn = sum(p.numel() for p in red_cnn.parameters())
print(f"\nTotal: {total_cnn:,} parámetros")

Unos **20 500 parámetros**: ¡cinco veces menos que los 101 770 de la red de RN 4! La mayoría están en la última capa `Linear`; las dos capas convolucionales juntas tienen menos de 5000.

Para comparar en igualdad de condiciones, construimos también la red de RN 4 y usamos las mismas funciones de entrenamiento:

In [ ]:
torch.manual_seed(0)
red_mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
print(f"Red de RN 4: {sum(p.numel() for p in red_mlp.parameters()):,} parámetros")

torch.manual_seed(0)
cargador_entreno = DataLoader(entreno, batch_size=64, shuffle=True)
cargador_prueba = DataLoader(prueba, batch_size=1000)
criterio = nn.CrossEntropyLoss()

def entrenar_una_epoca(red, cargador, optimizador):
    red.train()
    suma = 0
    for X, y in cargador:
        perdida = criterio(red(X), y)
        optimizador.zero_grad()
        perdida.backward()
        optimizador.step()
        suma += perdida.item() * len(X)
    return suma / len(cargador.dataset)

def exactitud(red, cargador, desplazamiento=0):
    red.eval()
    correctos = 0
    with torch.no_grad():
        for X, y in cargador:
            X = torch.roll(X, shifts=desplazamiento, dims=3)   # corre la imagen a la derecha
            correctos += (red(X).argmax(dim=1) == y).sum().item()
    return correctos / len(cargador.dataset)

Son las funciones de RN 4 con dos cambios: el optimizador ahora es un argumento (porque entrenaremos dos redes), y `exactitud` puede **correr** las imágenes antes de evaluarlas (lo usaremos en la sección 6).

¡A entrenar las dos redes, 3 épocas cada una! (La red convolucional hace muchas más cuentas por imagen, así que tarda más: uno o dos minutos en CPU.)

In [ ]:
import time

historia = {}
for nombre, red in [("RN 4 (densa)", red_mlp), ("convolucional", red_cnn)]:
    optimizador = optim.SGD(red.parameters(), lr=0.1)
    historia[nombre] = []
    inicio = time.time()
    for epoca in range(1, 4):
        perd = entrenar_una_epoca(red, cargador_entreno, optimizador)
        exa = exactitud(red, cargador_prueba)
        historia[nombre].append(exa)
        print(f"{nombre:14} época {epoca}: pérdida = {perd:.4f}   exactitud en prueba = {exa:.2%}   ({time.time() - inicio:.0f} s)")

In [ ]:
plt.figure(figsize=(5.5, 3.3))
for nombre, exas in historia.items():
    plt.plot(range(1, 4), [100 * e for e in exas], "o-", label=nombre)
plt.xticks(range(1, 4))
plt.xlabel("época"); plt.ylabel("% de aciertos en prueba")
plt.title("Densa contra convolucional")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

for nombre, red in [("RN 4 (densa)", red_mlp), ("convolucional", red_cnn)]:
    errores = round((1 - historia[nombre][-1]) * len(prueba))
    print(f"{nombre:14}: {errores} errores en 10 000 imágenes de prueba")

Con **cinco veces menos parámetros**, la red convolucional se equivoca en bastantes menos dígitos. (Tus números pueden variar un poco.)

✏️ *¿Por qué crees que una red con menos parámetros puede funcionar mejor? Piensa en qué sabe la convolución «de fábrica» que la red densa tiene que aprender a la fuerza.*

## 6. ¿Y si corremos el dígito?

Repitamos el experimento que puso en aprietos a la red de RN 4: correr **todas** las imágenes de prueba de 0 a 5 píxeles a la derecha.

In [ ]:
desplazamientos = range(6)
resultados = {nombre: [exactitud(red, cargador_prueba, d) for d in desplazamientos]
              for nombre, red in [("RN 4 (densa)", red_mlp), ("convolucional", red_cnn)]}

print("px   densa   convolucional")
for d in desplazamientos:
    print(f"{d:2}   {resultados['RN 4 (densa)'][d]:6.1%}   {resultados['convolucional'][d]:6.1%}")

plt.figure(figsize=(5.5, 3.3))
for nombre, exas in resultados.items():
    plt.plot(desplazamientos, [100 * e for e in exas], "o-", label=nombre)
plt.xlabel("píxeles corridos a la derecha"); plt.ylabel("% de aciertos")
plt.title("Robustez al desplazamiento")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

La red convolucional **aguanta mucho mejor** los desplazamientos: sus filtros encuentran los trazos donde estén, y el pooling perdona los movimientos pequeños. Pero no es perfecta: con desplazamientos grandes también cae, porque la última capa `Linear` todavía mira **en qué zona** del mapa apareció cada trazo. En la próxima lección veremos cómo el **aumento de datos** ayuda aún más.

## 7. ¿Qué aprendieron los filtros?

Los 16 filtros de la primera capa empezaron al azar. Veamos en qué se convirtieron después de entrenar:

In [ ]:
pesos = red_cnn[0].weight.detach()          # forma (16, 1, 3, 3)
fig, ejes = plt.subplots(2, 8, figsize=(12, 4.2))
for k, eje in enumerate(ejes.flat):
    lim = pesos[k, 0].abs().max()
    eje.imshow(pesos[k, 0], cmap="RdBu", vmin=-lim, vmax=lim)
    eje.set_title(f"filtro {k}")
    eje.set_xticks([]); eje.set_yticks([])
plt.suptitle("Los 16 filtros aprendidos de la primera capa (rojo = negativo, azul = positivo)")
plt.tight_layout()
plt.show()

Son pequeños, así que cuesta leerlos. Es más claro ver **qué hace** cada filtro con un dígito: los 16 mapas de características de la primera capa.

In [ ]:
imagen_prueba = prueba[0][0].unsqueeze(0)    # forma (1, 1, 28, 28); es un 7
with torch.no_grad():
    mapas = red_cnn[1](red_cnn[0](imagen_prueba))[0]    # Conv2d y ReLU: forma (16, 28, 28)

fig, ejes = plt.subplots(2, 9, figsize=(14, 3.6))
ejes[0, 0].imshow(imagen_prueba[0, 0], cmap="gray_r"); ejes[0, 0].set_title("entrada")
ejes[1, 0].axis("off")
for k in range(16):
    eje = ejes.flat[k + 1 + (k >= 8)]
    eje.imshow(mapas[k], cmap="magma")
    eje.set_title(f"mapa {k}")
for eje in ejes.flat:
    eje.set_xticks([]); eje.set_yticks([])
plt.tight_layout()
plt.show()

Algunos mapas se iluminan en los bordes de arriba del trazo, otros en los de abajo, otros en los lados o en todo el trazo. Algunos mapas quedan casi negros: ese filtro no encontró en este 7 lo que busca. ¡La red **descubrió sola** filtros parecidos a los que diseñamos a mano en la sección 2!

✏️ *Busca un mapa que resalte sobre todo la raya horizontal de arriba del 7, y otro que resalte la raya inclinada. Mira su filtro en la figura anterior: ¿tiene sentido?*

## 8. Errores comunes

**Error 1: olvidar la dimensión del lote o del canal.** `nn.Conv2d` espera 4 dimensiones (lote, canales, alto, ancho) o al menos 3 (canales, alto, ancho). Una imagen de 28 × 28 «pelada» no sirve:

In [ ]:
imagen_sola = torch.tensor(digito)            # forma (28, 28)
red_cnn(imagen_sola)

El remedio es `imagen_sola.reshape(1, 1, 28, 28)` o `imagen_sola.unsqueeze(0).unsqueeze(0)`.

**Error 2: equivocarse en el tamaño de la capa `Linear`.** Si quitamos el segundo pooling, el mapa final mide 32 × 14 × 14 = 6272, no 32 × 7 × 7 = 1568:

In [ ]:
sin_pool = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 10),
)
sin_pool(torch.zeros(1, 1, 28, 28))

El mensaje dice `1x6272 and 1568x10`: la red entrega 6272 números y la capa esperaba 1568. El truco de la sección 5 (pasar una imagen de ceros capa por capa e imprimir `x.shape`) evita adivinar.

**Error 3: olvidar el padding.** Sin `padding=1`, cada convolución de 3 × 3 recorta un píxel por cada lado: 28 → 26 → 13 → 11 → 5. El cálculo de la capa final cambia.

**Error 4: confundir canales con filtros.** En `nn.Conv2d(16, 32, 3)` hay 32 filtros, y **cada uno** tiene 16 × 3 × 3 pesos, uno por cada canal que llega. Por eso la segunda capa tiene 32 · (16 · 9 + 1) = 4640 parámetros y no 32 · 10.

**Error 5: esperar que la red convolucional sea rápida en CPU.** Tiene menos parámetros, pero cada filtro se aplica en cientos de posiciones: hace **más cuentas** por imagen. Con muchas épocas conviene usar la GPU de Colab (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …aplicas `filtro_vertical` (sección 1) a una imagen **toda llena de unos**, `np.ones((6, 6))`?
2. …cambias el signo del filtro, `-filtro_vertical`, y lo aplicas a la imagen de la raya? ¿Qué pasa con los $-3$ y los $+3$?
3. …usas `nn.Conv2d(1, 16, kernel_size=5)` sin padding con una imagen de 28 × 28? ¿Qué forma tiene la salida? ¿Cuántos parámetros tiene la capa?
4. …aplicas `nn.MaxPool2d(2)` a un tensor de forma `(1, 16, 7, 7)`? (7 es impar.)
5. …le das a la red convolucional entrenada el dígito de prueba **volteado** de izquierda a derecha, `torch.flip(prueba[0][0], dims=[2])`? ¿Sigue reconociendo el 7?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Convolución a mano.** Calcula en tu cuaderno la convolución de esta imagen de 4 × 4 con el filtro `[[1, 1, 1], [0, 0, 0], [-1, -1, -1]]` (bordes horizontales). La salida mide 2 × 2. Compruébalo con la función `convolucion`.

$$\begin{bmatrix}1&1&1&1\\1&1&1&1\\0&0&0&0\\0&0&0&0\end{bmatrix}$$

**Ejercicio 2 (fácil). Tamaños y parámetros.** Una foto a color de 32 × 32 tiene 3 canales (rojo, verde, azul). Le aplicamos `nn.Conv2d(3, 10, kernel_size=5)` sin padding. Calcula a mano la forma de la salida y el número de parámetros. Compruébalo en PyTorch con `torch.zeros(1, 3, 32, 32)`.

**Ejercicio 3 (medio). Inventa un filtro.** Diseña un filtro de 3 × 3 que detecte **puntos aislados**: que dé un número grande cuando el píxel del centro es claro y sus 8 vecinos son oscuros. Pruébalo con una imagen de 7 × 7 que tenga un punto suelto y una raya, y comprueba que el punto da el valor más alto.

**Ejercicio 4 (medio). ¿Cuántos filtros?** Cambia la red convolucional para que la primera capa tenga 4 filtros y la segunda 8 (recuerda cambiar también la capa `Linear`). ¿Cuántos parámetros tiene? Entrénala **una** época y compara su exactitud con la red de la sección 5 después de una época.

**Ejercicio 5 (avanzado). Una tercera capa.** Agrega un tercer bloque `Conv2d(32, 64, 3, padding=1)`, `ReLU`, `MaxPool2d(2)`. ¿Qué tamaño tiene el mapa al final (recuerda que 7 es impar)? Ajusta la capa `Linear`, cuenta los parámetros, entrena 3 épocas y compara la exactitud y la robustez al desplazamiento con la red de dos bloques.

## 🏆 Reto opcional: ropa en lugar de dígitos

`torchvision` trae **Fashion-MNIST**: 70 000 imágenes de 28 × 28 de prendas (camisetas, pantalones, zapatos, bolsos…), con 10 clases, igual que MNIST pero bastante más difícil.

```python
entreno_ropa = datasets.FashionMNIST("datos", train=True, download=True, transform=transforms.ToTensor())
prueba_ropa = datasets.FashionMNIST("datos", train=False, download=True, transform=transforms.ToTensor())
```

1. Muestra 16 imágenes con su número de clase. Las clases son: 0 camiseta, 1 pantalón, 2 suéter, 3 vestido, 4 abrigo, 5 sandalia, 6 camisa, 7 tenis, 8 bolso, 9 bota.
2. Entrena la red densa de RN 4 y la red convolucional de esta lección, 3 épocas cada una. ¿Cuál gana y por cuánto? ¿La diferencia es mayor o menor que con MNIST?
3. Con la matriz de confusión, ¿qué prendas confunde más la red convolucional? ¿Te sorprende?

## 🎬 Cierre

Hoy aprendiste que:
- Una **convolución** desliza un **filtro** pequeño por toda la imagen y, en cada posición, suma los productos casilla por casilla. El resultado es un **mapa de características** que dice dónde está lo que el filtro busca.
- Un filtro de 3 × 3 tiene solo 9 pesos y los **comparte** en todas las posiciones: por eso encuentra un trazo esté donde esté.
- El tamaño de la salida es $n + 2p - k + 1$, y el **max pooling** lo reduce a la mitad, resumiendo cada zona y tolerando movimientos pequeños.
- Una red convolucional con unos 20 500 parámetros le gana a la red densa de RN 4 (101 770 parámetros) y aguanta mucho mejor los desplazamientos.
- La red **aprende sola** filtros parecidos a los detectores de bordes que diseñamos a mano.

**Para pensar:** hoy entrenamos con el mismo optimizador de siempre (SGD) y solo 3 épocas. ¿Habrá formas de entrenar más rápido y mejor? ¿Qué pasa si la red **memoriza** los datos de entrenamiento? En la próxima lección veremos los trucos que usan quienes entrenan redes de verdad: el optimizador **Adam**, el **dropout**, el **aumento de datos** y una buena **inicialización** de los pesos. 🛠️